In [9]:
import pickle
from pathlib import Path
import pandas as pd


def get_experiment_file(results_dir="../results", file_name=None):
    """
    If file_name is provided:
      - if it's a path that exists, use it
      - else treat it as a filename under results_dir
    Else:
      - fall back to the latest .pkl in results_dir
    """
    folder = Path(results_dir)

    # 1) user specified a file
    if file_name is not None:
        p = Path(file_name)
        if p.exists():
            return p
        p = folder / file_name
        if p.exists():
            return p
        raise FileNotFoundError(f"Pickle not found: {file_name} (looked in {folder.resolve()})")

    # 2) fallback: latest
    pickle_files = list(folder.glob("*.pkl"))
    if not pickle_files:
        raise FileNotFoundError(f"No experiment files found in {folder.resolve()}")
    return max(pickle_files, key=lambda p: p.stat().st_mtime)

def print_experiment_summary(pickle_path, results_dir="../results"):
    """
    Load + print summary for a specific pickle.

    Accepts:
      - full/relative path: "../results/very_long_name.pkl"
      - short filename: "very_long_name.pkl" (resolved inside results_dir)
    """
    p = Path(pickle_path)
    print(f"📦 Loading experiment data from: {pickle_path}\n")

    with open(p, "rb") as f:
        exp_data = pickle.load(f) 
        
    # 1. Print Configurations
    print("="*60)
    print(" 🛠️ EXPERIMENT CONFIGURATION")
    print("="*60)
    print(f"Config Profile : {exp_data.get('cfg_name')}")
    print(f"Params Profile : {exp_data.get('prm_name')}")
    print(f"\nFull Config Object:\n{exp_data.get('cfg')}")
    print(f"\nFull Params Object:\n{exp_data.get('prm')}")
    print("\n")

    # 2. Print Strategy Performance Summary
    print("="*60)
    print(" 📈 PERFORMANCE SUMMARY")
    print("="*60)
    
    stats = exp_data.get('stats', {})
    meta = exp_data.get('meta', {})
    
    summary_rows = []
    for strat_name, strat_stats in stats.items():
        row = {
            "Pair": strat_name,
            "Total Ret": f"{strat_stats.get('Total Return', 0):.2%}",
            "Sharpe": round(strat_stats.get('Sharpe (rf=0)', 0), 2),
            "Max DD": f"{strat_stats.get('Max Drawdown', 0):.2%}",
            "Win Rate": f"{strat_stats.get('Win Rate', 0):.1%}", 
            "IS/OS Split": meta.get(strat_name, {}).get('n0_pct', 'N/A')
        }
        summary_rows.append(row)
        
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows)
        summary_df = summary_df.sort_values(by="Sharpe", ascending=False)
        print(summary_df.to_string(index=False))
    else:
        print("No strategy statistics found.")
    print("\n")

    # 3. Print Diagnostics
    print("="*60)
    print(" 🔍 DATA CLEANING DIAGNOSTICS (Top Performing Pair)")
    print("="*60)
    
    if summary_rows and meta:
        top_strat = summary_df.iloc[0]["Pair"]
        print(f"Diagnostics for: {top_strat}\n")
        
        strat_meta = meta.get(top_strat, {})
        diags = strat_meta.get("diagnostics", {})
        
        print(f"Initial Kalman Priors:")
        print(f"  - Beta0: {strat_meta.get('beta0')}")
        print(f"  - Q Est: {strat_meta.get('Q')}")
        print(f"  - R Est: {strat_meta.get('R')}\n")
        
        print(f"Data Cleaning Phase:")
        print(f"  - X Clean Diag: {diags.get('x_clean_diag')}")
        print(f"  - Y Clean Diag: {diags.get('y_clean_diag')}")
        print(f"  - Resample Diag: {diags.get('x_resample_diag')}")
        print(f"  - Pair Build Diag: {diags.get('pair_build_diag', 'Pending Implementation')}")



In [10]:
def list_pickles(results_dir="../results"):
    folder = Path(results_dir)
    for p in sorted(folder.glob("*.pkl"), key=lambda x: x.stat().st_mtime, reverse=True):
        print(p.name)



In [11]:
print_experiment_summary(r"C:\Users\Admin\PycharmProjects\Kalman-Pair-Trading\results\20260310-151045_S1_post_ffill_unlimited_P0_baseline.pkl")

📦 Loading experiment data from: C:\Users\Admin\PycharmProjects\Kalman-Pair-Trading\results\20260310-151045_S1_post_ffill_unlimited_P0_baseline.pkl

 🛠️ EXPERIMENT CONFIGURATION
Config Profile : S1_post_ffill_unlimited
Params Profile : P0_baseline

Full Config Object:
{'resample_method': 'vwap', 'pre_merge_fill': 'none', 'post_merge_fill': 'ffill', 'dropna_after_merge': True, 'pre_merge_limit': None, 'post_merge_limit': None}

Full Params Object:
{'name': 'P0_baseline', 'window_pct_R': 0.02, 'P0': 1.0, 'n0_beta_0': 0.0375, 'trade_by': 'posterior_spread', 'z_sco_win': 60, 'entry_z': 2.0, 'exit_z': 0.5}


 📈 PERFORMANCE SUMMARY
  Pair Total Ret  Sharpe  Max DD Win Rate  IS/OS Split
B_vs_C   183.94%    3.82 -35.83%     0.0%       0.0375
A_vs_C   137.31%    3.62 -36.61%     0.0%       0.0375
C_vs_B    79.19%    3.44 -22.40%     0.0%       0.0375
A_vs_B   111.03%    2.95 -89.37%     0.0%       0.0375
B_vs_A    71.54%    2.57 -83.80%     0.0%       0.0375
C_vs_A    45.84%    2.54 -25.33%     

In [6]:
import pickle
from pathlib import Path
from datetime import datetime

results_dir = Path("../results")
all_files = list(results_dir.glob("*.pkl"))

print(f"Found {len(all_files)} total .pkl files in: {results_dir.resolve()}")

# Print all files to see if timestamps are messing with the sort
for p in all_files:
    mtime = datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    print(f" - {p.name} | Modified: {mtime}")

if all_files:
    latest_file = max(all_files, key=lambda p: p.stat().st_mtime)
    print(f"\n📁 SELECTED FILE: {latest_file.name}")
    
    with open(latest_file, "rb") as f:
        raw_data = pickle.load(f)
        
    print(f"\n🧬 ROOT DATA TYPE: {type(raw_data)}")
    
    if isinstance(raw_data, tuple):
        print(f"📏 Tuple Length: {len(raw_data)}")
        for i, item in enumerate(raw_data):
            print(f"  -> Item {i} Type: {type(item)}")
    elif isinstance(raw_data, dict):
        print(f"🔑 Dict Keys: {list(raw_data.keys())}")

Found 1 total .pkl files in: C:\Users\Admin\PycharmProjects\Kalman-Pair-Trading\results
 - 20260310-144343_S1_post_ffill_unlimited_P0_baseline.pkl | Modified: 2026-03-10 14:43:43

📁 SELECTED FILE: 20260310-144343_S1_post_ffill_unlimited_P0_baseline.pkl

🧬 ROOT DATA TYPE: <class 'dict'>
🔑 Dict Keys: ['cfg_name', 'prm_name', 'cfg', 'prm', 'results_equity', 'stats', 'meta']
